## Local Inference on GPU
Model page: https://huggingface.co/facebook/nllb-200-distilled-600M

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/nllb-200-distilled-600M)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
# Warning: Pipeline type "translation" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# pip install "transformers<5.0.0"
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def translate_text(text, target_lang="tam_Taml"):
    if not text.strip():
        return "Please enter some text."

    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(text, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
        max_new_tokens=200
    )

    translated_text = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    return translated_text

In [ ]:
result = translate_text("Hello, how are you?", "tam_Taml")
print("Tamil Translation:", result)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Tamil Translation: வணக்கம், எப்படி இருக்கிறீர்கள்?


In [ ]:
!pip install -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.7.1
    Uninstalling gradio_client-2.7.1:
      Successfully uninstalled gradio_client-2.7.1
  Attempting uninstall: gradio
    Found existing installation: gradio 6.29.0
    Uninstalling gradio-6.29.0:
      Successfully uninstalled gradio-6.29.0


In [ ]:

import gradio as gr
import torch

# -------------------- LANGUAGE OPTIONS --------------------
languages = {
    "English": "eng_Latn",
    "Tamil (தமிழ்)": "tam_Taml",
    "Hindi (हिन्दी)": "hin_Deva",
    "Telugu (తెలుగు)": "tel_Telu",
    "Malayalam (മലയാളം)": "mal_Mlym",
    "Kannada (ಕನ್ನಡ)": "kan_Knda",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "German": "deu_Latn",
    "Arabic": "arb_Arab",
    "Japanese": "jpn_Jpan",
    "Chinese (Simplified)": "zho_Hans",
    "Portuguese": "por_Latn",
    "Russian": "rus_Cyrl"
}

# -------------------- TRANSLATION FUNCTION --------------------
def translate_text(text, source_language, target_language):
    if not text or not text.strip():
        return "Please enter a sentence to translate."

    if source_language == target_language:
        return text

    tokenizer.src_lang = languages[source_language]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    target_token_id = tokenizer.convert_tokens_to_ids(
        languages[target_language]
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_new_tokens=200
        )

    return tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]


# -------------------- SWAP LANGUAGES --------------------
def swap_languages(source, target):
    return target, source


# -------------------- CUSTOM CSS --------------------
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

.gradio-container {
    font-family: 'Inter', sans-serif !important;
    background:
        radial-gradient(ellipse at 8% 15%, rgba(99,102,241,.12), transparent 35%),
        radial-gradient(ellipse at 92% 12%, rgba(56,189,248,.15), transparent 35%),
        linear-gradient(135deg, #f4f8ff, #f8f5ff 55%, #eefaff) !important;
    max-width: 100% !important;
}

.gradio-container .main {
    max-width: 1350px !important;
    margin: auto !important;
}

/* Header */
#topbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 18px 10px;
    border-bottom: 1px solid rgba(255,255,255,.8);
    margin-bottom: 25px;
}

.brand {
    font-size: 25px;
    font-weight: 800;
    color: #172554;
    letter-spacing: -1px;
}

.brand span {
    color: #6d28d9;
}

.top-note {
    font-size: 12px;
    color: #64748b;
}

/* Hero */
#hero {
    text-align: center;
    padding: 12px 10px 34px;
}

.hero-title {
    font-size: clamp(30px, 4vw, 49px);
    font-weight: 800;
    letter-spacing: -2px;
    line-height: 1.2;
    color: #10194b;
    margin: 0 0 15px;
}

.hero-title span {
    background: linear-gradient(90deg, #2563eb, #7c3aed, #a21caf);
    -webkit-background-clip: text;
    background-clip: text;
    color: transparent;
}

.hero-subtitle {
    max-width: 720px;
    margin: auto;
    font-size: 16px;
    line-height: 1.8;
    color: #52678e;
}

/* Main translator: no thick outer border or glow */
#translator-card {
    position: relative;
    padding: 34px;
    border: none !important;
    border-radius: 22px;
    background: rgba(255,255,255,.72) !important;
    box-shadow: 0 12px 35px rgba(70,100,180,.08);
    overflow: visible;
}

#translator-card:before {
    display: none !important;
    content: none !important;
}

/* Roomy language selectors */
#translator-card [data-testid="dropdown"] {
    width: 100% !important;
    min-height: 54px !important;
    border: 1px solid #d5e1ff !important;
    border-radius: 13px !important;
    background: rgba(255,255,255,.96) !important;
    box-shadow: 0 3px 10px rgba(70,100,180,.04);
}

#translator-card [data-testid="dropdown"] input {
    min-height: 44px !important;
    font-size: 15px !important;
}

#translator-card .row {
    gap: 18px !important;
}

.field-label {
    font-size: 14px !important;
    font-weight: 700 !important;
    color: #17316d !important;
}

/* Textboxes */
#translator-card input,
#translator-card textarea {
    border: 1px solid #d5e1ff !important;
    border-radius: 13px !important;
    background: rgba(255,255,255,.94) !important;
    box-shadow: 0 3px 10px rgba(70,100,180,.035);
}

#translator-card input:focus,
#translator-card textarea:focus {
    border-color: #818cf8 !important;
    box-shadow: 0 0 0 3px rgba(129,140,248,.14) !important;
}

/* Swap button */
#swap-btn button {
    width: 54px;
    height: 54px;
    border-radius: 50% !important;
    border: 1px solid #ddd6fe !important;
    background: linear-gradient(135deg, #fff, #f0edff) !important;
    color: #633cf0 !important;
    font-size: 25px !important;
    box-shadow: 0 4px 12px rgba(99,60,240,.10);
    margin-top: 22px;
    transition: transform .3s ease;
}

#swap-btn button:hover {
    transform: rotate(180deg);
}

/* Input text area */
#input-text textarea {
    min-height: 145px !important;
    padding: 18px !important;
    font-size: 15px !important;
    line-height: 1.8 !important;
}

/* Character counter */
#section-title {
    font-weight: 700;
    color: #19316d;
    font-size: 14px;
    margin: 12px 0;
}

/* Gradient translate button */
#translate-btn button {
    background: linear-gradient(100deg, #9333ea, #6366f1 52%, #1685f8) !important;
    color: white !important;
    border: none !important;
    border-radius: 50px !important;
    min-height: 52px;
    font-weight: 700 !important;
    font-size: 16px !important;
    box-shadow: 0 7px 18px rgba(99,102,241,.25);
    transition: all .25s ease;
}

#translate-btn button:hover {
    transform: translateY(-2px);
    box-shadow: 0 10px 25px rgba(99,102,241,.35);
}

/* Translation output */
#output-panel {
    border: 1px solid rgba(191,207,255,.8) !important;
    border-radius: 18px;
    background: rgba(245,248,255,.7) !important;
    padding: 17px;
    margin-top: 10px;
}

#output-text textarea {
    min-height: 85px !important;
    padding: 14px !important;
    background: rgba(255,255,255,.95) !important;
    font-size: 15px !important;
    line-height: 1.8 !important;
}

/* Popular language chips */
#popular-title {
    font-weight: 800;
    color: #19316d;
    font-size: 16px;
    margin: 28px 0 5px;
}

.section-subtitle {
    font-size: 12px;
    color: #64748b;
    margin-bottom: 13px;
}

#language-chips {
    display: flex;
    flex-wrap: wrap;
    gap: 9px;
    justify-content: center;
}

.lang-chip {
    border: 1px solid #dbe5ff;
    border-radius: 30px;
    background: rgba(255,255,255,.7);
    padding: 10px 16px;
    color: #264477;
    font-size: 12px;
    font-weight: 600;
}

/* Feature row */
#benefits {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
    margin-top: 28px;
    padding: 22px 10px;
    border-top: 1px solid rgba(200,215,255,.5);
}

.benefit {
    display: flex;
    gap: 12px;
    align-items: center;
    padding: 8px;
}

.benefit-icon {
    width: 42px;
    height: 42px;
    flex-shrink: 0;
    display: grid;
    place-items: center;
    color: white;
    border-radius: 50%;
    font-size: 20px;
    background: linear-gradient(135deg, #8b5cf6, #3b82f6);
    box-shadow: 0 5px 15px rgba(99,102,241,.2);
}

.benefit strong {
    display: block;
    font-size: 12px;
    color: #172554;
    margin-bottom: 4px;
}

.benefit small {
    display: block;
    font-size: 10px;
    line-height: 1.5;
    color: #64748b;
}

#footer {
    text-align: center;
    color: #7283a8;
    font-size: 11px;
    padding: 12px;
}

/* Mobile responsive layout */
@media (max-width: 700px) {
    #translator-card {
        padding: 16px !important;
        border-radius: 18px;
    }

    #translator-card .row {
        gap: 10px !important;
    }

    #benefits {
        grid-template-columns: repeat(2, 1fr);
    }

    .top-note {
        display: none;
    }

    .hero-title {
        letter-spacing: -1px;
    }
}
"""


# -------------------- WEBPAGE DESIGN --------------------
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="AI Translator | Universal Language Translator"
) as demo:

    gr.HTML("""
    <div id="topbar">
        <div class="brand">🌐 <span>AI</span> Translator</div>
        <div class="top-note">
            One Language &nbsp; • &nbsp; Many Cultures
            &nbsp; • &nbsp; Endless Possibilities
        </div>
    </div>
    """)

    gr.HTML("""
    <div id="hero">
        <h1 class="hero-title">
            Universal <span>Language Translator</span>
        </h1>
        <p class="hero-subtitle">
            ✨ Break language barriers with the power of AI.
            Translate your text between languages quickly and easily.
        </p>
    </div>
    """)

    with gr.Group(elem_id="translator-card"):

        # Wider language selectors with a centered swap button
        with gr.Row(equal_height=True):
            with gr.Column(scale=12, min_width=180):
                source_dropdown = gr.Dropdown(
                    choices=list(languages.keys()),
                    value="English",
                    label="✦ From Language",
                    elem_classes=["field-label"]
                )

            with gr.Column(scale=1, min_width=60, elem_id="swap-btn"):
                swap_button = gr.Button("⇄", size="lg")

            with gr.Column(scale=12, min_width=180):
                target_dropdown = gr.Dropdown(
                    choices=list(languages.keys()),
                    value="Tamil (தமிழ்)",
                    label="✦ To Language",
                    elem_classes=["field-label"]
                )

        input_text = gr.Textbox(
            lines=4,
            max_lines=8,
            label="✎ Enter your text",
            placeholder="Type or paste a sentence in your language...",
            elem_id="input-text"
        )

        character_count = gr.Markdown(
            "✦ Enter text above to begin your translation.",
            elem_id="section-title"
        )

        with gr.Row():
            with gr.Column(scale=1):
                pass
            with gr.Column(scale=3, elem_id="translate-btn"):
                translate_button = gr.Button(
                    "✨  Translate  →",
                    variant="primary"
                )
            with gr.Column(scale=1):
                pass

        with gr.Group(elem_id="output-panel"):
            with gr.Row():
                gr.Markdown("✦ **Translated text**")
                copy_button = gr.Button("📋 Copy", size="sm")

            output_text = gr.Textbox(
                lines=3,
                max_lines=8,
                show_label=False,
                placeholder="Your translation will appear here...",
                interactive=False,
                elem_id="output-text"
            )

    # Live character counter
    input_text.input(
        fn=lambda text: f"📝 {len(text or '')} characters",
        inputs=input_text,
        outputs=character_count
    )

    # Translate button
    translate_button.click(
        fn=translate_text,
        inputs=[input_text, source_dropdown, target_dropdown],
        outputs=output_text
    )

    # Swap source and target languages
    swap_button.click(
        fn=swap_languages,
        inputs=[source_dropdown, target_dropdown],
        outputs=[source_dropdown, target_dropdown]
    )

    # Copy translated text
    copy_button.click(
        fn=None,
        inputs=output_text,
        outputs=None,
        js="(text) => { if (text) navigator.clipboard.writeText(text); }"
    )

    # Popular languages and feature highlights
    gr.HTML("""
    <div id="popular-title">✨ Popular Languages</div>
    <div class="section-subtitle">
        Quickly translate between commonly used languages
    </div>

    <div id="language-chips">
        <div class="lang-chip">🇬🇧 English</div>
        <div class="lang-chip">🇮🇳 Tamil</div>
        <div class="lang-chip">🇮🇳 Hindi</div>
        <div class="lang-chip">🇮🇳 Telugu</div>
        <div class="lang-chip">🇮🇳 Malayalam</div>
        <div class="lang-chip">🇫🇷 French</div>
        <div class="lang-chip">🇪🇸 Spanish</div>
        <div class="lang-chip">🇩🇪 German</div>
    </div>

    <div id="benefits">
        <div class="benefit">
            <div class="benefit-icon">ϟ</div>
            <div>
                <strong>Fast &amp; Simple</strong>
                <small>Translate sentences in seconds.</small>
            </div>
        </div>

        <div class="benefit">
            <div class="benefit-icon">◎</div>
            <div>
                <strong>Multiple Languages</strong>
                <small>Powered by the NLLB model.</small>
            </div>
        </div>

        <div class="benefit">
            <div class="benefit-icon">✓</div>
            <div>
                <strong>Hugging Face AI</strong>
                <small>AI-powered translation.</small>
            </div>
        </div>

        <div class="benefit">
            <div class="benefit-icon">♡</div>
            <div>
                <strong>Easy to Use</strong>
                <small>A simple, friendly interface.</small>
            </div>
        </div>
    </div>

    <div id="footer">
        Built with ❤️ using Hugging Face Transformers
    </div>
    """)

    gr.ClearButton(
        [input_text, output_text],
        value="🗑 Clear text"
    )


# -------------------- LAUNCH WEBPAGE --------------------
demo.launch(share=True)


/tmp/ipykernel_978/965186819.py:369: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71a6fd47a9409a3fdf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
